In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Download data
ticker = "NVDA"  # or "AMD"
data = yf.download(ticker, start="2018-01-01", end="2025-01-01")

# Fix columns if needed
if isinstance(data.columns, pd.MultiIndex):
    data.columns = data.columns.get_level_values(0)

data = data[['Close']].dropna()

[*********************100%***********************]  1 of 1 completed


In [2]:
# Parameters (keep simple)
ma_window = 20
momentum_window = 5
alpha = 0.15  # tune manually (0.1–0.2 range)

# Features
data['MA'] = data['Close'].rolling(ma_window).mean()
data['Residual'] = data['Close'] - data['MA']
data['Residual_Pred'] = data['Residual'].shift(1)
data['Momentum'] = data['Close'] - data['Close'].shift(momentum_window)

# Prediction
data['Prediction'] = (
    data['MA'] 
    + data['Residual_Pred'] 
    + alpha * data['Momentum']
)

# Directions
data['Actual_Dir'] = np.sign(data['Close'].diff())
data['Pred_Dir'] = np.sign(data['Prediction'].diff())

data = data.dropna()

In [3]:
# Split
split = int(len(data) * 0.8)
train = data.iloc[:split]
test = data.iloc[split:]

# Model performance
rmse = np.sqrt(mean_squared_error(test['Close'], test['Prediction']))
mae = mean_absolute_error(test['Close'], test['Prediction'])
acc = (test['Actual_Dir'] == test['Pred_Dir']).mean()

# Naive baseline
test['Naive'] = test['Close'].shift(1)
test = test.dropna()

naive_acc = (np.sign(test['Close'].diff()) == np.sign(test['Naive'].diff())).mean()

print("════════════════════════════════════")
print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")
print(f"Directional Accuracy: {acc:.2%}")
print("──────────────")
print(f"Naive Accuracy: {naive_acc:.2%}")
print(f"Improvement: {(acc - naive_acc):.2%}")
print("════════════════════════════════════")

════════════════════════════════════
RMSE: 2.76
MAE: 1.93
Directional Accuracy: 56.03%
──────────────
Naive Accuracy: 49.28%
Improvement: 6.75%
════════════════════════════════════


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_23984\873220602.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test['Naive'] = test['Close'].shift(1)


In [4]:
# --- Copy of data ---
df = data.copy()

# --- Parameters (same as baseline) ---
ma_window = 20
momentum_window = 5
alpha = 0.15

# --- Base features ---
df['MA'] = df['Close'].rolling(ma_window).mean()
df['Residual'] = df['Close'] - df['MA']
df['Residual_Pred'] = df['Residual'].shift(1)
df['Momentum'] = df['Close'] - df['Close'].shift(momentum_window)

# --- NEW FEATURE: Volatility ---
df['Volatility'] = df['Close'].rolling(20).std()

# --- Baseline Prediction ---
df['Pred_Base'] = (
    df['MA'] 
    + df['Residual_Pred'] 
    + alpha * df['Momentum']
)

# --- Enhanced Prediction (with volatility adjustment) ---
beta = 0.05  # small weight to avoid overfitting

df['Pred_Vol'] = (
    df['MA'] 
    + df['Residual_Pred'] 
    + alpha * df['Momentum']
    - beta * df['Volatility']   # dampens prediction in high volatility
)

# --- Directions ---
df['Actual'] = np.sign(df['Close'].diff())
df['Dir_Base'] = np.sign(df['Pred_Base'].diff())
df['Dir_Vol'] = np.sign(df['Pred_Vol'].diff())

df = df.dropna()

# --- Train/Test Split ---
split = int(len(df) * 0.8)
test = df.iloc[split:].copy()

# --- Accuracy ---
base_acc = (test['Actual'] == test['Dir_Base']).mean()
vol_acc = (test['Actual'] == test['Dir_Vol']).mean()

# --- Naive baseline ---
test['Naive'] = test['Close'].shift(1)
test = test.dropna()
naive_acc = (np.sign(test['Close'].diff()) == np.sign(test['Naive'].diff())).mean()

# --- Results ---
print("════════════════════════════════════")
print(f"Naive Accuracy      : {naive_acc:.2%}")
print("──────────────")
print(f"Baseline Accuracy   : {base_acc:.2%}")
print(f"+ Volatility Accuracy: {vol_acc:.2%}")
print("──────────────")
print(f"Change vs Baseline  : {(vol_acc - base_acc):+.2%}")
print("════════════════════════════════════")

════════════════════════════════════
Naive Accuracy      : 49.27%
──────────────
Baseline Accuracy   : 56.10%
+ Volatility Accuracy: 55.81%
──────────────
Change vs Baseline  : -0.29%
════════════════════════════════════


In [5]:
# --- Copy data ---
df = data.copy()

# --- Parameters ---
ma_window = 20
alpha = 0.15

# --- Base features ---
df['MA'] = df['Close'].rolling(ma_window).mean()
df['Residual'] = df['Close'] - df['MA']
df['Residual_Pred'] = df['Residual'].shift(1)

# --- Single Momentum (baseline) ---
df['Mom_5'] = df['Close'] - df['Close'].shift(5)

# --- Multi-Momentum (new feature) ---
df['Mom_10'] = df['Close'] - df['Close'].shift(10)
df['Momentum_Multi'] = 0.5 * df['Mom_5'] + 0.5 * df['Mom_10']

# --- Predictions ---
df['Pred_Base'] = (
    df['MA'] 
    + df['Residual_Pred'] 
    + alpha * df['Mom_5']
)

df['Pred_Multi'] = (
    df['MA'] 
    + df['Residual_Pred'] 
    + alpha * df['Momentum_Multi']
)

# --- Directions ---
df['Actual'] = np.sign(df['Close'].diff())
df['Dir_Base'] = np.sign(df['Pred_Base'].diff())
df['Dir_Multi'] = np.sign(df['Pred_Multi'].diff())

df = df.dropna()

# --- Train/Test Split ---
split = int(len(df) * 0.8)
test = df.iloc[split:].copy()

# --- Accuracy ---
base_acc = (test['Actual'] == test['Dir_Base']).mean()
multi_acc = (test['Actual'] == test['Dir_Multi']).mean()

# --- Naive baseline ---
test['Naive'] = test['Close'].shift(1)
test = test.dropna()
naive_acc = (np.sign(test['Close'].diff()) == np.sign(test['Naive'].diff())).mean()

# --- Results ---
print("════════════════════════════════════")
print(f"Naive Accuracy        : {naive_acc:.2%}")
print("──────────────")
print(f"Baseline Accuracy     : {base_acc:.2%}")
print(f"Multi-Momentum Accuracy: {multi_acc:.2%}")
print("──────────────")
print(f"Change vs Baseline    : {(multi_acc - base_acc):+.2%}")
print("════════════════════════════════════")

════════════════════════════════════
Naive Accuracy        : 49.27%
──────────────
Baseline Accuracy     : 56.10%
Multi-Momentum Accuracy: 55.81%
──────────────
Change vs Baseline    : -0.29%
════════════════════════════════════


In [6]:
# --- Copy data ---
df = data.copy()

# --- Parameters ---
ma_window = 20
momentum_window = 5
alpha = 0.15
gamma = 0.1  # small weight for distance

# --- Features ---
df['MA'] = df['Close'].rolling(ma_window).mean()
df['Residual'] = df['Close'] - df['MA']
df['Residual_Pred'] = df['Residual'].shift(1)
df['Momentum'] = df['Close'] - df['Close'].shift(momentum_window)

# --- NEW FEATURE: Distance from MA ---
df['Distance'] = df['Close'] - df['MA']

# --- Predictions ---
df['Pred_Base'] = (
    df['MA'] 
    + df['Residual_Pred'] 
    + alpha * df['Momentum']
)

# Mean reversion adjustment (subtract distance)
df['Pred_Dist'] = (
    df['MA'] 
    + df['Residual_Pred'] 
    + alpha * df['Momentum']
    - gamma * df['Distance']
)

# --- Directions ---
df['Actual'] = np.sign(df['Close'].diff())
df['Dir_Base'] = np.sign(df['Pred_Base'].diff())
df['Dir_Dist'] = np.sign(df['Pred_Dist'].diff())

df = df.dropna()

# --- Split ---
split = int(len(df) * 0.8)
test = df.iloc[split:].copy()

# --- Accuracy ---
base_acc = (test['Actual'] == test['Dir_Base']).mean()
dist_acc = (test['Actual'] == test['Dir_Dist']).mean()

# --- Naive ---
test['Naive'] = test['Close'].shift(1)
test = test.dropna()
naive_acc = (np.sign(test['Close'].diff()) == np.sign(test['Naive'].diff())).mean()

# --- Results ---
print("════════════════════════════════════")
print(f"Naive Accuracy        : {naive_acc:.2%}")
print("──────────────")
print(f"Baseline Accuracy     : {base_acc:.2%}")
print(f"+ Distance Accuracy   : {dist_acc:.2%}")
print("──────────────")
print(f"Change vs Baseline    : {(dist_acc - base_acc):+.2%}")
print("════════════════════════════════════")

════════════════════════════════════
Naive Accuracy        : 49.27%
──────────────
Baseline Accuracy     : 56.10%
+ Distance Accuracy   : 52.91%
──────────────
Change vs Baseline    : -3.20%
════════════════════════════════════


In [7]:
df = data.copy()

ma_window = 20
momentum_window = 5
alpha = 0.15

# Features
df['MA'] = df['Close'].rolling(ma_window).mean()
df['Residual'] = df['Close'] - df['MA']
df['Residual_Pred'] = df['Residual'].shift(1)

# OLD momentum
df['Momentum'] = df['Close'] - df['Close'].shift(momentum_window)

# NEW normalized momentum
df['Momentum_norm'] = (
    (df['Close'] - df['Close'].shift(momentum_window)) 
    / df['Close'].shift(momentum_window)
)

# Predictions
df['Pred_Base'] = df['MA'] + df['Residual_Pred'] + alpha * df['Momentum']

df['Pred_Norm'] = (
    df['MA'] 
    + df['Residual_Pred'] 
    + alpha * df['Momentum_norm'] * df['Close']  # scale back
)

# Directions
df['Actual'] = np.sign(df['Close'].diff())
df['Dir_Base'] = np.sign(df['Pred_Base'].diff())
df['Dir_Norm'] = np.sign(df['Pred_Norm'].diff())

df = df.dropna()

split = int(len(df) * 0.8)
test = df.iloc[split:].copy()

base_acc = (test['Actual'] == test['Dir_Base']).mean()
norm_acc = (test['Actual'] == test['Dir_Norm']).mean()

test['Naive'] = test['Close'].shift(1)
test = test.dropna()
naive_acc = (np.sign(test['Close'].diff()) == np.sign(test['Naive'].diff())).mean()

print("════════════════════════════════════")
print(f"Naive Accuracy        : {naive_acc:.2%}")
print("──────────────")
print(f"Baseline Accuracy     : {base_acc:.2%}")
print(f"Normalized Momentum   : {norm_acc:.2%}")
print("──────────────")
print(f"Change vs Baseline    : {(norm_acc - base_acc):+.2%}")
print("════════════════════════════════════")

════════════════════════════════════
Naive Accuracy        : 49.27%
──────────────
Baseline Accuracy     : 56.10%
Normalized Momentum   : 55.81%
──────────────
Change vs Baseline    : -0.29%
════════════════════════════════════


In [8]:
df = data.copy()

ma_window = 20
momentum_window = 5
alpha = 0.15
delta = 0.1  # RSI weight

# --- Base features ---
df['MA'] = df['Close'].rolling(ma_window).mean()
df['Residual'] = df['Close'] - df['MA']
df['Residual_Pred'] = df['Residual'].shift(1)
df['Momentum'] = df['Close'] - df['Close'].shift(momentum_window)

# --- RSI ---
delta_price = df['Close'].diff()
gain = (delta_price.where(delta_price > 0, 0)).rolling(14).mean()
loss = (-delta_price.where(delta_price < 0, 0)).rolling(14).mean()
rs = gain / (loss + 1e-9)
df['RSI'] = 100 - (100 / (1 + rs))

# Normalize RSI (center around 0)
df['RSI_norm'] = (df['RSI'] - 50) / 50

# --- Predictions ---
df['Pred_Base'] = df['MA'] + df['Residual_Pred'] + alpha * df['Momentum']

df['Pred_RSI'] = (
    df['MA'] 
    + df['Residual_Pred'] 
    + alpha * df['Momentum']
    + delta * df['RSI_norm'] * df['Close']
)

# --- Directions ---
df['Actual'] = np.sign(df['Close'].diff())
df['Dir_Base'] = np.sign(df['Pred_Base'].diff())
df['Dir_RSI'] = np.sign(df['Pred_RSI'].diff())

df = df.dropna()

split = int(len(df) * 0.8)
test = df.iloc[split:].copy()

base_acc = (test['Actual'] == test['Dir_Base']).mean()
rsi_acc = (test['Actual'] == test['Dir_RSI']).mean()

test['Naive'] = test['Close'].shift(1)
test = test.dropna()
naive_acc = (np.sign(test['Close'].diff()) == np.sign(test['Naive'].diff())).mean()

print("════════════════════════════════════")
print(f"Naive Accuracy        : {naive_acc:.2%}")
print("──────────────")
print(f"Baseline Accuracy     : {base_acc:.2%}")
print(f"+ RSI Accuracy        : {rsi_acc:.2%}")
print("──────────────")
print(f"Change vs Baseline    : {(rsi_acc - base_acc):+.2%}")
print("════════════════════════════════════")

════════════════════════════════════
Naive Accuracy        : 49.27%
──────────────
Baseline Accuracy     : 56.10%
+ RSI Accuracy        : 67.15%
──────────────
Change vs Baseline    : +11.05%
════════════════════════════════════
